In [1]:
import numpy as np
from scipy.optimize import curve_fit

In [3]:
#Estimating paramters of a non-linear regression by non-linear least squares and GMM
def model_func(X, a, b):
    x1, x2 = X
    return np.exp(b * x1) + a * x2


#Generating synthetic data
np.random.seed(0)
X = np.random.rand(100, 2)
a_true = 2.5
b_true = 0.5
y_data = model_func((X[:, 0], X[:, 1]), a_true, b_true) + np.random.normal(size=100)*0.1
#First curve fit with non-linear least squares
popt, pcov = curve_fit(model_func, (X[:, 0], X[:, 1]), y_data, p0=(1.0, 0.1))

print("Estimated parameters (non-linear least squares):", popt)
print("True parameters:", [a_true, b_true])
print("standard errors:", np.sqrt(np.diag(pcov)))

Estimated parameters (non-linear least squares): [2.51115225 0.47813619]
True parameters: [2.5, 0.5]
standard errors: [0.02347427 0.01572107]


In [6]:
#Now we use the moment condition E[ \nabla_theta g(x, theta) (y - g(x, theta)) ] = 0
def moment_conditions(params, X, y):
    a, b = params
    residuals = y - model_func((X[:, 0], X[:, 1]), a, b)
    grad_a = -X[:, 1]
    grad_b = -X[:, 0] * np.exp(b * X[:, 0])
    moments = np.vstack((grad_a * residuals, grad_b * residuals)).T
    return moments.mean(axis=0)

from scipy.optimize import minimize
def gmm_objective(params, X, y):
    moments = moment_conditions(params, X, y)
    return moments @ moments  # Simple quadratic form with identity weighting

initial_guess = [1.0, 0.1]
result = minimize(gmm_objective, initial_guess, args=(X, y_data))
print("Estimated parameters (GMM):", result.x)
print("True parameters:", [a_true, b_true])

Estimated parameters (GMM): [2.51113531 0.47814806]
True parameters: [2.5, 0.5]


In [10]:
import numpy as np
from scipy.optimize import minimize

# -----------------------------
# 1. Model: choose a test g(x, beta)
#    (replace g_fun and grad_g_fun with your own)
# -----------------------------
def g_fun(x, beta):
    """
    Nonlinear mean function g(x_t, beta).
    Here: g(x,b) = tanh(x b)
    x : (T, p)
    beta : (p,)
    returns: (T,)
    """
    z = x @ beta
    return np.tanh(z)


def grad_g_fun(x, beta):
    """
    Gradient wrt beta of g(x_t, beta).
    For g(x,b) = tanh(x b), dg/dz = 1 - tanh(z)^2, and dz/db = x_t.
    So grad g_t(beta) = (1 - tanh(z_t)^2) * x_t
    x : (T, p)
    beta : (p,)
    returns: (T, p)
    """
    z = x @ beta
    factor = 1.0 - np.tanh(z) ** 2  # (T,)
    return factor[:, None] * x      # (T, p)


# -----------------------------
# 2. Moment function
# -----------------------------
def moments(beta, x, y):
    """
    Stack the two sets of moments:

    m1_t(beta) = grad_beta g(x_t,beta) * (y_t - g(x_t,beta))   [NLLS conditions]
    m2_t(beta) = x_t * (y_t - x_t' beta)                       [OLS conditions]

    Returns matrix m with shape (T, 2p).
    """
    beta = np.asarray(beta)
    T, p = x.shape

    g = g_fun(x, beta)                  # (T,)
    grad_g = grad_g_fun(x, beta)        # (T, p)

    # NLLS residuals and moments
    res_nl = y - g                      # (T,)
    m1 = grad_g * res_nl[:, None]       # (T, p)

    # OLS residuals and moments
    res_ols = y - x @ beta              # (T,)
    m2 = x * res_ols[:, None]           # (T, p)

    # Stack: (T, 2p)
    m = np.concatenate([m1, m2], axis=1)
    return m


def gmm_objective(beta, x, y, W):
    """
    GMM objective: J(beta) = mbar(beta)' W mbar(beta)
    where mbar(beta) is the sample mean of the stacked moments.
    """
    m = moments(beta, x, y)
    mbar = m.mean(axis=0)               # (2p,)
    return mbar @ W @ mbar


# -----------------------------
# 3. Numerical Jacobian of mbar(beta)
#    needed for robust GMM covariance
# -----------------------------
def jacobian_mbar(beta, x, y, h=1e-5):
    """
    Numerical derivative of mbar(beta) wrt beta.
    Returns D of shape (2p, p).
    """
    beta = np.asarray(beta)
    k = beta.size

    m0 = moments(beta, x, y).mean(axis=0)  # (2p,)
    D = np.zeros((m0.size, k))
    for j in range(k):
        bp = beta.copy()
        bp[j] += h
        mp = moments(bp, x, y).mean(axis=0)
        D[:, j] = (mp - m0) / h
    return D


# -----------------------------
# 4. Simulate data to test the code
# -----------------------------
np.random.seed(0)
T = 1000
p = 2

X = np.random.normal(size=(T, p))
beta_true = np.array([0.5, -0.3])

# Data generating process: y_t = g(X_t, beta_true) + epsilon_t
eps = 0.1 * np.random.normal(size=T)
y = g_fun(X, beta_true) + eps

# -----------------------------
# 5. Two-step GMM estimation
# -----------------------------
# First step: identity weighting
W0 = np.eye(2 * p)
beta_init = np.zeros(p)

res1 = minimize(gmm_objective, beta_init, args=(X, y, W0), method='BFGS')
beta_1step = res1.x

# Optimal weighting matrix
m_1step = moments(beta_1step, X, y)
S = np.cov(m_1step, rowvar=False, bias=True)       # (2p, 2p), long-run covariance of moments
W_opt = np.linalg.inv(S)

# Second step: re-estimate with optimal W
res2 = minimize(gmm_objective, beta_1step, args=(X, y, W_opt), method='BFGS')
beta_hat = res2.x

# -----------------------------
# 6. Robust GMM covariance for beta_hat
# -----------------------------
m_hat = moments(beta_hat, X, y)
mbar_hat = m_hat.mean(axis=0)
S_hat = np.cov(m_hat, rowvar=False, bias=True)
W = np.linalg.inv(S_hat)

D = jacobian_mbar(beta_hat, X, y)  # (2p, p)

# Sandwich formula: Var(beta) = (D' W D)^(-1) (D' W S W D) (D' W D)^(-1) / T
A = D.T @ W @ D
B = D.T @ W @ S_hat @ W @ D

Var_beta = np.linalg.inv(A) @ B @ np.linalg.inv(A) / T
se_beta = np.sqrt(np.diag(Var_beta))

# -----------------------------
# 7. Print results
# -----------------------------
print("True beta:      ", beta_true)
print("1-step GMM beta:", beta_1step)
print("2-step GMM beta:", beta_hat)
print("Std. errors:    ", se_beta)


True beta:       [ 0.5 -0.3]
1-step GMM beta: [ 0.40713533 -0.24650563]
2-step GMM beta: [ 0.44823637 -0.27060049]
Std. errors:     [0.00444686 0.00385972]
